# Corrective RAG (CRAG): When Retrieval Fails, Fix It

## The Problem: Standard RAG Blindly Trusts Retrieval

Standard RAG has a critical vulnerability: it **unconditionally trusts** whatever the retriever returns. The pipeline is simple — retrieve `k` documents, stuff them into a prompt, generate. But what happens when the retriever returns irrelevant garbage?

**The answer propagates garbage into the generation.** The LLM faithfully synthesizes an answer from wrong context, producing a confident-sounding but factually incorrect response. The user has no idea.

This is the **silent failure mode** of RAG: no error, no warning, just a plausible-looking wrong answer. The failure rate increases with:
- **Out-of-distribution queries** — questions the knowledge base wasn't designed for
- **Sparse coverage topics** — the knowledge base has tangential but not direct information
- **Ambiguous queries** — where retrieval returns partially-relevant documents

CRAG addresses this by adding a **self-assessment layer** between retrieval and generation.

## What is Corrective RAG (CRAG)?

**Corrective Retrieval-Augmented Generation** was introduced by [Yan et al., 2024](https://arxiv.org/abs/2401.15884). The core insight is deceptively simple:

> *Before using retrieved documents, evaluate whether they actually answer the question. If they don't, take corrective action.*

CRAG introduces a **retrieval evaluator** that classifies retrieval quality into three categories:

| Verdict | Confidence | Action |
|---------|-----------|--------|
| **CORRECT** | High (score ≥ 0.7) | Proceed with retrieved documents |
| **AMBIGUOUS** | Medium (0.3 ≤ score < 0.7) | Refine query, re-retrieve, synthesize |
| **INCORRECT** | Low (score < 0.3) | Abandon retrieval, use fallback sources |

This transforms RAG from a **feed-forward pipeline** into a **self-correcting system** with feedback loops.

## The CRAG Pipeline: Six-Stage Architecture

```
   ┌──────────┐     ┌──────────────┐     ┌──────────────┐
   │  Query   │────▶│  Retrieve k  │────▶│  Confidence  │
   │          │     │  Documents   │     │   Scorer     │
   └──────────┘     └──────────────┘     └──────┬───────┘
                                                │
                    ┌────────────────────────────┼────────────────────┐
                    │                            │                    │
                    ▼                            ▼                    ▼
             ┌──────────┐               ┌───────────────┐    ┌──────────────┐
             │ CORRECT  │               │  AMBIGUOUS    │    │  INCORRECT   │
             │ score≥0.7│               │  0.3≤s<0.7   │    │  score<0.3   │
             └────┬─────┘               └──────┬────────┘    └──────┬───────┘
                  │                            │                    │
                  ▼                            ▼                    ▼
          Extract key info         Query decomposition      Fallback retrieval
          from top docs            + re-retrieve +          (different index,
                  │                synthesize partial        broader search)
                  │                answers                         │
                  │                      │                         │
                  └──────────────────────┼─────────────────────────┘
                                         ▼
                                  ┌──────────────┐
                                  │   Generate    │
                                  │   Answer      │
                                  └──────────────┘
```

Each stage:
1. **Retrieve** — Standard vector similarity search (FAISS + BGE embeddings)
2. **Score** — LLM evaluates relevance of each document to the query (1–5 scale)
3. **Decide** — Map scores to CORRECT / AMBIGUOUS / INCORRECT
4. **Correct path** — Extract key information, pass directly to generator
5. **Ambiguous path** — Decompose query into sub-questions, re-retrieve, merge partial answers
6. **Incorrect path** — Use fallback knowledge source (secondary index, broader corpus)

## Setup

We load **Qwen3-14B** (4-bit NF4 quantization) for LLM tasks and **BGE-base-en-v1.5** for dense embeddings. Model weights are cached to Google Drive so re-runs are fast.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch
!pip install -q sentence-transformers faiss-cpu numpy

import torch, json, textwrap, re, numpy as np
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=1024, temperature=0.6, thinking=False):
    """Generate a response with Qwen3. Returns answer string (thinking disabled by default for CRAG)."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=thinking,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens, do_sample=True, top_k=20,
        temperature=temperature, top_p=0.8,
    )
    output_ids = model.generate(**inputs, **gen_kwargs)
    new_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME} (4-bit NF4)")
print(f"  Device: {model.device}")

In [ ]:
# Load BGE embedding model for semantic search
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
embedder = SentenceTransformer(EMBED_MODEL, cache_folder=CACHE_DIR)

print(f"✓ Embedding model loaded: {EMBED_MODEL}")
print(f"  Embedding dim: {embedder.get_sentence_embedding_dimension()}")

## Building the Knowledge Base

We create a synthetic knowledge base with **intentional gaps**. This is crucial for demonstrating CRAG — we need topics with:
- **Dense coverage** (many relevant chunks) → triggers CORRECT path
- **Sparse/tangential coverage** (partial info) → triggers AMBIGUOUS path
- **No coverage** (completely off-topic queries) → triggers INCORRECT path

Our knowledge base covers **renewable energy** in depth, has **tangential mentions** of battery storage, and says **nothing** about quantum computing.

In [ ]:
# Synthetic knowledge base — designed with intentional coverage gaps
KNOWLEDGE_BASE = [
    # === Dense coverage: Solar Energy (5 chunks) ===
    "Solar photovoltaic (PV) cells convert sunlight directly into electricity using the "
    "photovoltaic effect. Modern monocrystalline silicon cells achieve 22-24% efficiency "
    "in commercial panels. The global installed solar capacity exceeded 1,200 GW in 2023.",

    "The cost of solar PV has dropped 90% since 2010, from roughly $0.36/kWh to under "
    "$0.05/kWh in optimal locations. This dramatic price reduction is driven by manufacturing "
    "scale, improved cell efficiency, and supply chain optimization in China.",

    "Perovskite solar cells represent the next frontier in photovoltaics. These cells use "
    "a perovskite-structured compound as the light-absorbing layer. Lab efficiencies have "
    "reached 33.7% in tandem configurations with silicon. Key challenges include long-term "
    "stability and lead toxicity in the perovskite layer.",

    "Solar panel degradation rates average 0.5-0.8% per year for modern panels, meaning a "
    "panel retains roughly 80-87% of its original output after 25 years. The main degradation "
    "mechanisms are potential-induced degradation (PID), light-induced degradation (LID), and "
    "UV-induced encapsulant browning.",

    "Utility-scale solar farms use single-axis trackers to follow the sun east-to-west, "
    "increasing energy yield by 15-25% compared to fixed-tilt systems. The largest solar farm, "
    "Bhadla Solar Park in India, has a capacity of 2.25 GW across 14,000 acres.",

    # === Dense coverage: Wind Energy (4 chunks) ===
    "Onshore wind turbines have grown dramatically in size. Modern turbines have rotor "
    "diameters exceeding 170 meters and hub heights of 120+ meters. Larger rotors capture "
    "more energy from lower wind speeds, making wind viable in more locations.",

    "Offshore wind has become a major growth sector. The global offshore wind capacity "
    "reached 75 GW in 2023. Floating offshore wind platforms are being developed for deeper "
    "waters (60-200m depth), unlocking vast new resource areas off the coasts of Japan, "
    "California, and the Mediterranean.",

    "The capacity factor of onshore wind typically ranges from 25-45%, while offshore wind "
    "achieves 40-55% due to stronger, more consistent winds at sea. The levelized cost of "
    "wind energy has fallen below $0.04/kWh in the best locations.",

    "Wind energy faces intermittency challenges. Production varies with wind speed and "
    "follows seasonal patterns. Grid operators manage this through geographic diversification, "
    "forecasting improvements, and flexible backup generation.",

    # === Sparse coverage: Battery Storage (1 chunk — intentionally thin) ===
    "Grid-scale battery storage is growing rapidly. Lithium-ion batteries dominate the market "
    "with costs around $150/kWh. Alternative chemistries like sodium-ion and iron-air are "
    "being developed for longer-duration storage applications.",

    # === Sparse coverage: Hydrogen Economy (1 chunk — tangential) ===
    "Green hydrogen, produced via electrolysis powered by renewable energy, is being explored "
    "as a storage medium and fuel for hard-to-electrify sectors like shipping and steelmaking. "
    "Current electrolyzer costs remain high at $500-1,400/kW.",

    # === Moderate coverage: Grid Integration (2 chunks) ===
    "Integrating high shares of variable renewables requires grid modernization. Smart grids "
    "use real-time sensors, automated switching, and demand response to balance supply and "
    "demand. Countries like Denmark manage 50%+ wind penetration through interconnections.",

    "Virtual power plants (VPPs) aggregate distributed energy resources — rooftop solar, "
    "home batteries, EVs, and flexible loads — into a coordinated system that can provide "
    "grid services. VPPs are growing rapidly in Australia and Germany.",

    # === No coverage: Quantum Computing, Ancient History (zero chunks) ===
    # Intentionally absent — queries about these should trigger INCORRECT path
]

print(f"✓ Knowledge base: {len(KNOWLEDGE_BASE)} chunks")
print(f"  Dense: Solar (5), Wind (4)")
print(f"  Sparse: Battery (1), Hydrogen (1)")
print(f"  Moderate: Grid (2)")
print(f"  Missing: Quantum computing, Ancient history")

## Building the FAISS Index

We encode all knowledge chunks into dense vectors using BGE and build a FAISS index for similarity search. We store the original texts alongside the index for retrieval.

In [ ]:
# Encode chunks and build FAISS index
embeddings = embedder.encode(KNOWLEDGE_BASE, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product = cosine sim (normalized vectors)
index.add(embeddings)

print(f"✓ FAISS index built: {index.ntotal} vectors, dimension {dimension}")
print(f"  Index type: Flat Inner Product (exact cosine similarity)")

## Stage 1: The Retriever

The retriever is standard — encode the query, search the FAISS index, return top-k chunks with their similarity scores. Nothing fancy here; the intelligence comes later in the evaluator.

In [ ]:
def retrieve(query, k=3):
    """Retrieve top-k chunks from FAISS. Returns list of (text, score) tuples."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(KNOWLEDGE_BASE):
            results.append((KNOWLEDGE_BASE[idx], float(scores[0][i])))
    return results

# Quick test
test_results = retrieve("How efficient are solar panels?")
print("Test query: 'How efficient are solar panels?'")
for i, (text, score) in enumerate(test_results):
    print(f"\n  [{i+1}] Score: {score:.4f}")
    print(f"      {text[:100]}...")

## Stage 2: Confidence Scoring — The Heart of CRAG

This is where CRAG diverges from standard RAG. Instead of blindly using retrieved documents, we ask the LLM to **evaluate** how well each document answers the query.

### Why Use the LLM as Evaluator?

Vector similarity scores (cosine similarity from FAISS) measure **topical relatedness**, not **answer relevance**. A document about "solar panel manufacturing in China" has high cosine similarity to "how efficient are solar panels" because both are about solar panels — but it doesn't actually answer the question about efficiency.

The LLM evaluator performs a deeper semantic check: *Does this document contain information that would help answer this specific question?*

### Scoring Rubric

We use a 1–5 scale:
- **5**: Document directly and completely answers the question
- **4**: Document contains strong relevant information
- **3**: Document is partially relevant but incomplete
- **2**: Document is tangentially related
- **1**: Document is irrelevant to the question

In [ ]:
EVAL_PROMPT = """You are a retrieval quality evaluator. Given a QUESTION and a DOCUMENT, rate how relevant the document is for answering the question.

Score on a scale of 1-5:
5 = Document directly and completely answers the question
4 = Document contains strong relevant information for answering
3 = Document is partially relevant but incomplete
2 = Document is tangentially related
1 = Document is irrelevant

Respond with ONLY a JSON object: {{"score": <number>, "reason": "<brief explanation>"}}

QUESTION: {question}
DOCUMENT: {document}"""

def score_relevance(question, document):
    """Use LLM to score document relevance on 1-5 scale. Returns (score, reason)."""
    prompt = EVAL_PROMPT.format(question=question, document=document)
    msgs = [{"role": "user", "content": prompt}]
    raw = generate(msgs, max_new_tokens=150, temperature=0.1)
    # Parse JSON from response
    try:
        # Find JSON in response
        match = re.search(r'\{[^}]+\}', raw)
        if match:
            data = json.loads(match.group())
            return int(data.get("score", 1)), data.get("reason", "")
    except (json.JSONDecodeError, ValueError):
        pass
    # Fallback: try to find a digit
    digits = re.findall(r'[1-5]', raw)
    return (int(digits[0]), raw) if digits else (1, raw)

# Test the scorer
test_doc = KNOWLEDGE_BASE[0]  # Solar PV chunk
score, reason = score_relevance("How efficient are solar panels?", test_doc)
print(f"Question: 'How efficient are solar panels?'")
print(f"Document: '{test_doc[:80]}...'")
print(f"Score: {score}/5")
print(f"Reason: {reason}")

## Stage 3: The Decision Gate

The decision gate maps the confidence scores to one of three actions. We score all retrieved documents and use the **best score** to decide the path:

```
best_score = max(scores across all retrieved docs)

if best_score >= 4:     → CORRECT    (normalized ≥ 0.7)
elif best_score >= 2:   → AMBIGUOUS  (normalized 0.3–0.7)
else:                   → INCORRECT  (normalized < 0.3)
```

We normalize the 1–5 scale to 0–1 for the thresholds: `normalized = (score - 1) / 4`

### Why Three Categories, Not Two?

A binary relevant/irrelevant gate would be simpler, but the **ambiguous** category is crucial. Many real queries fall in the gray zone — the retriever finds *something* related but not a direct answer. The ambiguous path tries to **salvage partial information** through query refinement rather than discarding it entirely.

In [ ]:
def classify_retrieval(question, retrieved_docs):
    """Score all documents and classify retrieval quality.
    Returns (verdict, scored_docs) where scored_docs is [(text, sim_score, llm_score, reason), ...]."""
    scored = []
    for text, sim_score in retrieved_docs:
        llm_score, reason = score_relevance(question, text)
        scored.append((text, sim_score, llm_score, reason))

    best_llm = max(s[2] for s in scored)
    normalized = (best_llm - 1) / 4.0  # Map 1-5 → 0-1

    if normalized >= 0.7:   # score >= 4
        verdict = "CORRECT"
    elif normalized >= 0.25: # score >= 2
        verdict = "AMBIGUOUS"
    else:                   # score == 1
        verdict = "INCORRECT"

    return verdict, scored

# Test on a well-covered topic
docs = retrieve("What is the efficiency of modern solar panels?")
verdict, scored = classify_retrieval("What is the efficiency of modern solar panels?", docs)
print(f"Query: 'What is the efficiency of modern solar panels?'")
print(f"Verdict: {verdict}")
for text, sim, llm, reason in scored:
    print(f"  sim={sim:.3f}  llm={llm}/5  | {text[:70]}...")

## Stage 4: The CORRECT Path — Extract and Generate

When retrieval is confident (score ≥ 4), we don't just dump the raw document into the prompt. We first **extract key information** relevant to the question. This serves two purposes:

1. **Noise reduction** — The chunk may contain irrelevant details alongside the answer
2. **Token efficiency** — Extracted key points are more concise than raw passages

This extraction step is what the original CRAG paper calls **knowledge refinement** — applied even on the correct path to maximize answer quality.

In [ ]:
def extract_key_info(question, document):
    """Extract key information from a document relevant to the question."""
    prompt = f"""Extract the key facts from this document that are relevant to answering the question.
Return only the relevant facts as a concise bullet list.

Question: {question}
Document: {document}

Key facts:"""
    msgs = [{"role": "user", "content": prompt}]
    return generate(msgs, max_new_tokens=300, temperature=0.2)


def handle_correct(question, scored_docs):
    """CORRECT path: use top-scoring documents to generate answer."""
    # Sort by LLM score (descending), take top 2
    ranked = sorted(scored_docs, key=lambda x: x[2], reverse=True)
    top_docs = ranked[:2]

    # Extract key info from each
    extracts = []
    for text, sim, llm, reason in top_docs:
        extracted = extract_key_info(question, text)
        extracts.append(extracted)

    context = "\n\n".join(extracts)
    return context, "direct_retrieval"

# Test key extraction
test_extract = extract_key_info(
    "How efficient are solar panels?",
    KNOWLEDGE_BASE[0]
)
print("Extracted key info:")
print(test_extract)

## Stage 5: The AMBIGUOUS Path — Knowledge Refinement

The ambiguous path is the most intellectually interesting part of CRAG. When retrieval returns *partially relevant* results, we don't discard them — we try to **squeeze more value** through a multi-step refinement:

### The Three-Step Refinement Process

1. **Query Decomposition** — Break the original question into 2–3 simpler sub-questions. The original query may be too complex or too specific for the knowledge base. Sub-questions target different aspects.

2. **Re-retrieve** — Search the index again with each sub-question. Different phrasings can surface different chunks that the original query missed.

3. **Synthesize** — Combine the partial answers from original retrieval + re-retrieval into a unified context.

### Why This Works

Consider: *"What is the long-term economic outlook for battery storage?"* Our knowledge base has one chunk about battery costs but nothing about economic projections. The original retrieval returns the battery chunk (tangentially relevant). Decomposition might produce:
- *"What is the current cost of battery storage?"* → matches the battery chunk well
- *"What trends affect energy storage economics?"* → might match the solar cost drop chunk

By casting a wider net with simpler sub-questions, we can often assemble a useful answer from partial information.

In [ ]:
def decompose_query(question):
    """Break a complex question into 2-3 simpler sub-questions."""
    prompt = f"""Break this question into 2-3 simpler sub-questions that, together, would help answer the original.
Return ONLY a JSON list of strings.

Question: {question}

Sub-questions:"""
    msgs = [{"role": "user", "content": prompt}]
    raw = generate(msgs, max_new_tokens=300, temperature=0.3)
    try:
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
    except json.JSONDecodeError:
        pass
    # Fallback: extract lines that look like questions
    lines = [l.strip().strip('"').strip("'").lstrip('0123456789.-) ') for l in raw.split('\n') if '?' in l]
    return lines[:3] if lines else [question]


def handle_ambiguous(question, scored_docs):
    """AMBIGUOUS path: decompose query, re-retrieve, synthesize."""
    print("    → Decomposing query...")
    sub_questions = decompose_query(question)
    print(f"    → Sub-questions: {sub_questions}")

    # Collect all relevant texts: original + re-retrieved
    all_texts = set()
    # Keep partially relevant original docs
    for text, sim, llm, reason in scored_docs:
        if llm >= 2:
            all_texts.add(text)

    # Re-retrieve for each sub-question
    for sq in sub_questions:
        new_docs = retrieve(sq, k=2)
        for text, score in new_docs:
            all_texts.add(text)

    print(f"    → Total unique chunks after re-retrieval: {len(all_texts)}")

    # Synthesize
    combined = "\n\n---\n\n".join(all_texts)
    synthesis_prompt = f"""Given the following information fragments, extract ALL relevant facts for answering: {question}

Information:
{combined}

Relevant facts (concise bullet list):"""
    msgs = [{"role": "user", "content": synthesis_prompt}]
    synthesized = generate(msgs, max_new_tokens=400, temperature=0.2)
    return synthesized, "ambiguous_refinement"

# Test decomposition
sub_qs = decompose_query("What is the long-term economic outlook for battery storage?")
print("Original: 'What is the long-term economic outlook for battery storage?'")
print("Sub-questions:")
for i, q in enumerate(sub_qs):
    print(f"  {i+1}. {q}")

## Stage 6: The INCORRECT Path — Fallback Retrieval

When the retriever completely fails (all scores ≤ 1), the documents are irrelevant and we need a completely different approach. In the original CRAG paper, this triggers **web search**. In our implementation, we simulate fallback options:

### Fallback Strategies

1. **Broader search** — Use the LLM to reformulate the query, then search a secondary/broader index
2. **Direct LLM knowledge** — Fall back to the model's parametric knowledge (no retrieval)
3. **Honest admission** — Acknowledge that the knowledge base doesn't cover this topic

We implement option 2 (LLM parametric knowledge) as our fallback, with an explicit disclaimer that the answer comes from the model's training data, not from our verified knowledge base.

In [ ]:
# Secondary/fallback knowledge base — broader but shallower coverage
FALLBACK_KB = [
    "Quantum computing uses qubits that can exist in superposition states, enabling "
    "parallel computation. Current quantum computers like IBM's Eagle processor have 127 "
    "qubits. Practical quantum advantage for real-world problems remains years away.",

    "Quantum error correction is a major challenge. Logical qubits require many physical "
    "qubits for error correction — estimates range from 1,000 to 10,000 physical qubits "
    "per logical qubit depending on the error rate.",

    "The Roman Empire at its height around 117 AD stretched from Britain to Mesopotamia, "
    "encompassing roughly 5 million square kilometers and 70 million people — about 25% of "
    "the world's population at the time.",

    "Ancient Egyptian civilization lasted over 3,000 years, from approximately 3100 BC to "
    "30 BC. The Great Pyramid of Giza was built around 2560 BC and remained the tallest "
    "human-made structure for nearly 4,000 years.",

    "Machine learning models require large datasets for training. Neural networks with "
    "billions of parameters, called large language models (LLMs), are trained on trillions "
    "of tokens of text data from the internet.",
]

# Build fallback FAISS index
fb_embeddings = embedder.encode(FALLBACK_KB, normalize_embeddings=True).astype("float32")
fallback_index = faiss.IndexFlatIP(fb_embeddings.shape[1])
fallback_index.add(fb_embeddings)


def retrieve_fallback(query, k=2):
    """Search the fallback knowledge base."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = fallback_index.search(q_emb, k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(FALLBACK_KB):
            results.append((FALLBACK_KB[idx], float(scores[0][i])))
    return results


def handle_incorrect(question, scored_docs):
    """INCORRECT path: try fallback index, then fall back to LLM knowledge."""
    print("    → Primary retrieval failed. Searching fallback index...")
    fallback_docs = retrieve_fallback(question, k=2)

    # Check if fallback has anything useful
    if fallback_docs:
        best_fb_text, best_fb_score = fallback_docs[0]
        fb_llm_score, fb_reason = score_relevance(question, best_fb_text)
        if fb_llm_score >= 3:
            print(f"    → Fallback hit! Score: {fb_llm_score}/5")
            context = "\n\n".join(text for text, _ in fallback_docs)
            return context, "fallback_retrieval"

    # Ultimate fallback: LLM parametric knowledge
    print("    → Fallback index also insufficient. Using LLM parametric knowledge.")
    parametric_prompt = f"""Answer this question using your general knowledge. Be concise and factual.
Note: This answer comes from the model's training data, not a verified knowledge base.

Question: {question}

Key facts:"""
    msgs = [{"role": "user", "content": parametric_prompt}]
    knowledge = generate(msgs, max_new_tokens=300, temperature=0.3)
    return knowledge, "llm_parametric"

# Test fallback retrieval
fb_results = retrieve_fallback("How many qubits do quantum computers have?")
print("Fallback test: 'How many qubits do quantum computers have?'")
for text, score in fb_results:
    print(f"  Score: {score:.3f} | {text[:80]}...")

## The Generator: Producing the Final Answer

After the decision gate routes to the appropriate handler and we have refined context, the generator produces the final answer. It includes a **source attribution** so the user knows where the information came from.

In [ ]:
SOURCE_LABELS = {
    "direct_retrieval": "📚 Source: Direct retrieval from knowledge base",
    "ambiguous_refinement": "🔄 Source: Refined retrieval (query decomposition + re-retrieval)",
    "fallback_retrieval": "🔀 Source: Fallback knowledge base",
    "llm_parametric": "🧠 Source: Model's parametric knowledge (not from knowledge base)",
}

def generate_answer(question, context, source_type):
    """Generate final answer from context."""
    prompt = f"""Answer the following question using ONLY the provided context. Be concise and factual.
If the context doesn't fully answer the question, say what you can and note the limitation.

Context:
{context}

Question: {question}

Answer:"""
    msgs = [{"role": "user", "content": prompt}]
    answer = generate(msgs, max_new_tokens=400, temperature=0.3)
    source_label = SOURCE_LABELS.get(source_type, source_type)
    return f"{answer}\n\n---\n{source_label}"

print("✓ Generator defined")
print("  Source labels:", list(SOURCE_LABELS.keys()))

## The Complete CRAG Pipeline

Now we assemble all the components into a single `crag_query()` function that orchestrates the entire flow: retrieve → evaluate → decide → handle → generate.

In [ ]:
def crag_query(question, verbose=True):
    """Full CRAG pipeline: retrieve, evaluate, decide, handle, generate."""
    if verbose:
        print(f"{'='*70}")
        print(f"QUESTION: {question}")
        print(f"{'='*70}")

    # Stage 1: Retrieve
    retrieved = retrieve(question, k=3)
    if verbose:
        print(f"\n[Stage 1] Retrieved {len(retrieved)} documents")
        for i, (text, score) in enumerate(retrieved):
            print(f"  Doc {i+1} (sim={score:.3f}): {text[:60]}...")

    # Stage 2 & 3: Evaluate and Decide
    verdict, scored = classify_retrieval(question, retrieved)
    if verbose:
        print(f"\n[Stage 2] Confidence scores:")
        for text, sim, llm, reason in scored:
            print(f"  LLM={llm}/5 (sim={sim:.3f}) | {reason[:60]}")
        print(f"\n[Stage 3] Decision: {verdict}")

    # Stage 4/5/6: Handle based on verdict
    if verdict == "CORRECT":
        if verbose: print("\n[Stage 4] CORRECT path: extracting key info...")
        context, source_type = handle_correct(question, scored)
    elif verdict == "AMBIGUOUS":
        if verbose: print("\n[Stage 5] AMBIGUOUS path: refining query...")
        context, source_type = handle_ambiguous(question, scored)
    else:
        if verbose: print("\n[Stage 6] INCORRECT path: trying fallback...")
        context, source_type = handle_incorrect(question, scored)

    if verbose:
        print(f"\n[Context assembled] ({len(context)} chars, source={source_type})")

    # Generate final answer
    if verbose: print("\n[Generating answer...]")
    answer = generate_answer(question, context, source_type)

    if verbose:
        print(f"\n{'─'*70}")
        print(f"ANSWER:\n{answer}")
        print(f"{'─'*70}")

    return {"question": question, "answer": answer, "verdict": verdict,
            "source_type": source_type, "context": context, "scored_docs": scored}

print("✓ CRAG pipeline assembled")

## Example 1: CORRECT Path — Well-Covered Topic

Our knowledge base has extensive coverage of solar energy. This query should retrieve highly relevant documents and proceed directly through the CORRECT path.

In [ ]:
result1 = crag_query("What is the efficiency of modern commercial solar panels and how has the cost changed?")

## Example 2: AMBIGUOUS Path — Sparse Coverage Topic

Our knowledge base has only **one chunk** about battery storage and **one chunk** about hydrogen. A question about "the future economics of energy storage" is partially covered — triggering the ambiguous path. Watch how CRAG decomposes the query and re-retrieves to assemble a more complete answer.

In [ ]:
result2 = crag_query("What is the long-term economic outlook for battery storage and hydrogen in the energy transition?")

## Example 3: INCORRECT Path — Completely Off-Topic Query

Our primary knowledge base has **nothing** about quantum computing. The retriever will return whatever is least-dissimilar (probably something about technology), but the confidence scorer should rate everything low, triggering the INCORRECT path.

In [ ]:
result3 = crag_query("How many qubits does IBM's latest quantum computer have and what are the error rates?")

## Comparison: Standard RAG vs. CRAG

To appreciate CRAG's value, let's implement a **naive standard RAG** pipeline and compare the outputs on the same queries. Standard RAG just retrieves and generates — no evaluation, no correction.

In [ ]:
def standard_rag(question, k=3):
    """Naive standard RAG: retrieve top-k, stuff into prompt, generate."""
    results = retrieve(question, k=k)
    context = "\n\n".join(text for text, score in results)

    prompt = f"""Answer the question using ONLY the provided context. Be concise.

Context:
{context}

Question: {question}

Answer:"""
    msgs = [{"role": "user", "content": prompt}]
    return generate(msgs, max_new_tokens=400, temperature=0.3)

print("✓ Standard RAG defined for comparison")

### Head-to-Head: Standard RAG vs. CRAG on Challenging Queries

Let's run both systems on three carefully chosen queries that test different failure modes.

In [ ]:
comparison_queries = [
    # Query 1: Well-covered (both should do fine)
    "What is the capacity factor of offshore wind turbines?",
    # Query 2: Sparse coverage (Standard RAG should struggle)
    "Compare the cost trends of lithium-ion vs sodium-ion batteries for grid storage.",
    # Query 3: Off-topic (Standard RAG will hallucinate from irrelevant context)
    "What were the main achievements of the Roman Empire?",
]

for i, q in enumerate(comparison_queries):
    print(f"\n{'='*70}")
    print(f"QUERY {i+1}: {q}")
    print(f"{'='*70}")

    # Standard RAG
    print("\n--- STANDARD RAG ---")
    std_answer = standard_rag(q)
    print(std_answer)

    # CRAG
    print("\n--- CRAG ---")
    crag_result = crag_query(q, verbose=False)
    print(crag_result["answer"])
    print(f"(Verdict: {crag_result['verdict']}, Source: {crag_result['source_type']})")

### Analysis: What the Comparison Reveals

| Query Type | Standard RAG | CRAG |
|-----------|-------------|------|
| **Well-covered** | ✅ Accurate answer from relevant docs | ✅ Same quality, with extraction step |
| **Sparse coverage** | ⚠️ Answer from tangential docs, may be incomplete or misleading | ✅ Detects ambiguity, decomposes query, assembles better answer |
| **Off-topic** | ❌ Synthesizes answer from irrelevant context = hallucination | ✅ Detects failure, falls back to alternative source with disclaimer |

The key insight: Standard RAG and CRAG perform **equally well** on easy queries (this is by design — CRAG adds overhead). CRAG's value shows on **hard queries** where retrieval fails silently.

**The cost of CRAG**: Each query requires `k` additional LLM calls for confidence scoring (one per retrieved document), plus potentially more calls for query decomposition in the ambiguous case. This is a latency/quality trade-off.

## Confidence Score Distribution: Understanding Retrieval Quality

Let's visualize how the confidence scorer discriminates between good and bad retrievals across multiple queries.

In [ ]:
# Score a batch of queries and visualize the distribution
test_queries = [
    ("How efficient are modern solar panels?", "well-covered"),
    ("What is the cost of offshore wind energy?", "well-covered"),
    ("How do virtual power plants work?", "moderate"),
    ("What are sodium-ion battery costs?", "sparse"),
    ("Explain quantum entanglement", "off-topic"),
    ("History of the Roman Empire", "off-topic"),
]

print(f"{'Query':<50} {'Best Sim':>8} {'Best LLM':>8} {'Verdict':>12}")
print("─" * 82)

for query, expected in test_queries:
    docs = retrieve(query, k=3)
    verdict, scored = classify_retrieval(query, docs)
    best_sim = max(s[1] for s in scored)
    best_llm = max(s[2] for s in scored)
    marker = "✓" if (
        (expected == "well-covered" and verdict == "CORRECT") or
        (expected == "sparse" and verdict == "AMBIGUOUS") or
        (expected == "moderate" and verdict in ["CORRECT", "AMBIGUOUS"]) or
        (expected == "off-topic" and verdict == "INCORRECT")
    ) else "✗"
    print(f"{query:<50} {best_sim:>8.3f} {best_llm:>8}/5 {verdict:>10} {marker}")

## Deep Dive: Why Cosine Similarity Alone Fails

A critical question: why not just use the FAISS cosine similarity scores as our confidence measure? Why add an expensive LLM evaluation step?

The answer: **cosine similarity measures topical overlap, not answer relevance.** Consider these cases:

| Query | Retrieved Doc | Cosine Sim | Actual Relevance |
|-------|--------------|-----------|------------------|
| "Cost of solar panels" | "Solar panel degradation rates..." | **High** | **Low** — talks about degradation, not cost |
| "Wind turbine size" | "Onshore wind turbines have grown..." | **High** | **High** — directly answers |
| "Quantum computing" | "Grid-scale battery storage..." | **Low** | **Low** — correct rejection |

The first case is the dangerous one: high cosine similarity but the document doesn't answer the question. Standard RAG would use this document and produce an irrelevant answer. The LLM evaluator catches this mismatch.

In [ ]:
# Demonstrate the cosine vs LLM score divergence
divergence_tests = [
    # (query, expected_behavior)
    ("What is the cost of solar energy?", "Should match cost chunk, not efficiency chunk"),
    ("How long do solar panels last?", "Should match degradation chunk"),
    ("What is green hydrogen used for?", "Sparse coverage — one chunk"),
]

for query, note in divergence_tests:
    print(f"\nQuery: {query}")
    print(f"Note: {note}")
    docs = retrieve(query, k=3)
    for text, sim in docs:
        llm_score, reason = score_relevance(query, text)
        match_status = "✓ ALIGNED" if (
            (sim > 0.6 and llm_score >= 4) or (sim < 0.4 and llm_score <= 2)
        ) else "⚠ DIVERGE"
        print(f"  {match_status} | sim={sim:.3f} llm={llm_score}/5 | {text[:60]}...")

## Production Considerations

### Latency Budget

CRAG adds significant latency compared to standard RAG:

| Stage | Standard RAG | CRAG (CORRECT) | CRAG (AMBIGUOUS) | CRAG (INCORRECT) |
|-------|-------------|---------------|-----------------|------------------|
| Retrieval | 1 search | 1 search | 1 + N searches | 1 + 1 search |
| LLM calls | 1 (generate) | k+1 (score+gen) | k+2+1 (score+decompose+gen) | k+1+1 (score+fallback+gen) |
| Total LLM | **1** | **k+1** | **k+3** | **k+2** |

With `k=3`, the CORRECT path uses 4× more LLM calls than standard RAG. The AMBIGUOUS path uses 6×.

### Optimization Strategies

1. **Batch scoring** — Score all k documents in a single LLM call (structured output)
2. **Lightweight evaluator** — Use a small fine-tuned classifier instead of the full LLM for scoring
3. **Caching** — Cache confidence scores for (query, document) pairs
4. **Adaptive k** — Start with k=1, only retrieve more if needed

### When to Use CRAG

CRAG is most valuable when:
- The knowledge base has **uneven coverage** (some topics deep, others shallow)
- **Incorrect answers are costly** (medical, legal, financial domains)
- Users ask **diverse queries** that may fall outside the knowledge base
- You need **transparent sourcing** (users need to know if the answer is verified)

In [ ]:
# Optimization: batch scoring all documents in one LLM call
def batch_score_relevance(question, documents):
    """Score multiple documents in a single LLM call for efficiency."""
    doc_list = "\n".join(
        f"DOCUMENT {i+1}: {doc[:200]}" for i, doc in enumerate(documents)
    )
    prompt = f"""Rate the relevance of each document to the question on a 1-5 scale.
Respond with ONLY a JSON list of objects: [{{"doc": 1, "score": N}}, ...]

QUESTION: {question}

{doc_list}"""
    msgs = [{"role": "user", "content": prompt}]
    raw = generate(msgs, max_new_tokens=300, temperature=0.1)
    try:
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        if match:
            results = json.loads(match.group())
            return [r.get("score", 1) for r in results]
    except (json.JSONDecodeError, ValueError):
        pass
    return [1] * len(documents)

# Compare: individual vs batch scoring
test_q = "How efficient are modern solar panels?"
test_docs = retrieve(test_q, k=3)
doc_texts = [t for t, s in test_docs]

# Batch score
batch_scores = batch_score_relevance(test_q, doc_texts)
print(f"Query: '{test_q}'")
print(f"\nBatch scores (single LLM call): {batch_scores}")
print("Individual scores (3 LLM calls):")
for doc, bs in zip(doc_texts, batch_scores):
    ind_score, _ = score_relevance(test_q, doc)
    print(f"  Batch={bs}/5  Individual={ind_score}/5  | {doc[:50]}...")

## Synthesis: CRAG as Part of a Larger System

### CRAG + Self-RAG: The Complete Self-Correcting Pipeline

CRAG and Self-RAG (Asai et al., 2023) are complementary techniques:

| Aspect | CRAG | Self-RAG |
|--------|------|----------|
| **Focus** | Retrieval quality | Generation quality |
| **Evaluates** | Are the documents relevant? | Is the answer supported by the documents? |
| **Corrects** | Re-retrieves or falls back | Re-generates with different context |
| **When** | Before generation | After generation |

A combined system would:
1. **CRAG** — Ensure retrieval quality before generation
2. **Generate** — Produce an answer from verified context
3. **Self-RAG** — Check if the answer is actually supported by the context
4. **Iterate** — If Self-RAG detects unsupported claims, flag them

### The Broader Picture: Agentic RAG

CRAG is one step toward **agentic RAG** — systems that reason about their own retrieval process:

```
Standard RAG  →  CRAG  →  Self-RAG  →  Agentic RAG
 (no eval)     (eval      (eval        (multi-step
               retrieval)  generation)   planning,
                                         tool use)
```

Each level adds intelligence — and cost. The engineering challenge is knowing when the extra intelligence is worth the extra latency.

## End-to-End Demo: All Three Paths in Action

Let's run a final comprehensive demo showing all three CRAG paths with detailed tracing.

In [ ]:
demo_queries = {
    "CORRECT": "What is the largest solar farm in the world and how big is it?",
    "AMBIGUOUS": "How do renewable energy systems handle the intermittency problem with long-duration storage?",
    "INCORRECT": "When was the Great Pyramid of Giza built and how tall is it?",
}

results = {}
for expected_path, query in demo_queries.items():
    print(f"\n{'▓'*70}")
    print(f"  EXPECTED PATH: {expected_path}")
    print(f"{'▓'*70}")
    result = crag_query(query, verbose=True)
    results[expected_path] = result
    actual = result['verdict']
    match = "✓ MATCHED" if actual == expected_path else f"✗ GOT {actual}"
    print(f"\n  Path verification: {match}")

In [ ]:
# Summary of all results
print("\n" + "=" * 80)
print("CRAG RESULTS SUMMARY")
print("=" * 80)
print(f"\n{'Query':<65} {'Verdict':<12} {'Source'}")
print("─" * 100)

all_results = [result1, result2, result3] + list(results.values())
for r in all_results:
    q = r['question'][:62] + '...' if len(r['question']) > 62 else r['question']
    print(f"{q:<65} {r['verdict']:<12} {r['source_type']}")

print("\n" + "=" * 80)
print("\nPath distribution:")
from collections import Counter
counts = Counter(r['verdict'] for r in all_results)
for verdict, count in counts.most_common():
    bar = "█" * (count * 5)
    print(f"  {verdict:<12} {bar} ({count})")

## Key Takeaways

### What We Built

A complete **Corrective RAG** system from scratch — no LangChain, no frameworks — using:
- **Qwen3-14B** (4-bit) as both the evaluator and generator
- **BGE-base-en-v1.5** for dense embeddings
- **FAISS** for vector search
- A **synthetic knowledge base** with intentional coverage gaps

### Core Principles

1. **Never blindly trust retrieval.** The retriever returns the *most similar* documents, not necessarily the *most relevant* ones.

2. **Evaluate before you generate.** A small investment in confidence scoring prevents large errors in generation.

3. **Three paths, not two.** The ambiguous middle ground is where CRAG adds the most value — salvaging partial information through query refinement.

4. **Fail transparently.** When retrieval truly fails, admit it and source the answer differently, with a clear disclaimer.

5. **The cost is real.** CRAG multiplies LLM calls by 4–6×. Use it when correctness matters more than latency.

### Looking Forward

CRAG is a stepping stone in the evolution of RAG systems:

- **Standard RAG** — Retrieve and generate (fast, fragile)
- **CRAG** — Evaluate retrieval, take corrective action (robust, slower)
- **Self-RAG** — Also evaluate generation quality (more robust, even slower)
- **Agentic RAG** — Multi-step planning, tool use, iterative refinement (most robust, most expensive)

The progression is toward systems that **reason about their own limitations** — a form of metacognition for AI.